# Watch for changes

**The job.** Check a thing on a schedule. Alert only when it actually changed.

The failure here is alerting every run. People turn those off, and then the one
that mattered goes unread.

A branch does the work: compare against last time, and take the alert path only
if something moved.

In [1]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

# These come from the library rather than being redefined in every notebook.
# They used to be thirty lines pasted into each one, which meant anyone copying
# a notebook to start a project got helpers that did not exist in browsergraph.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step
from browsergraph.quick import graph as _graph
from browsergraph.quick import subgraph  # noqa: F401  (used by later notebooks)

# The notebooks kept the older names, and `build` also prints what is wrong
# rather than raising — in a notebook the complaint is the lesson.
stage = step

def build(title, task, stages, nodes, edges=()):
    bench = _graph(title, task, stages, nodes, edges)
    print("problems:", problems(bench) or "none")
    return bench

print("ready")

ready


In [2]:
nodes = [
    node("read.now",    "read",    [],                  [("out", "Snapshot")]),
    node("read.last",   "recall",  [],                  [("out", "Snapshot")]),
    node("diff.two",    "compare", [("now", "Snapshot"), ("before", "Snapshot")],
         [("changed", "Diff"), ("same", "Diff")]),
    node("alert.write", "alert",   [("in", "Diff")],    [("out", "Receipt")],
         effects=("file.write",)),
    node("note.quiet",  "note",    [("in", "Diff")],    [("out", "Receipt")]),
    node("save.state",  "save",    [("in", "Snapshot")],[("out", "Receipt")],
         effects=("file.write",)),
]

stages = [
    stage("now",   "Read it now",   [], [("out", "Snapshot")], "read",   ["read.now"]),
    stage("last",  "What we saw",   [], [("out", "Snapshot")], "recall", ["read.last"]),
    StageDefinition(id="diff", name="Did it move?", kind="branch",
                    required_capabilities=("compare",),
                    inputs=(PortSpec("now", "Snapshot"), PortSpec("before", "Snapshot")),
                    outputs=(PortSpec("changed", "Diff"), PortSpec("same", "Diff")),
                    success="the two snapshots were compared",
                    candidates=("diff.two",)),
    stage("alert", "Raise an alert",[("in", "Diff")], [("out", "Receipt")], "alert", ["alert.write"]),
    stage("quiet", "Stay quiet",    [("in", "Diff")], [("out", "Receipt")], "note",  ["note.quiet"]),
    stage("save",  "Remember it",   [("in", "Snapshot")], [("out", "Receipt")], "save", ["save.state"]),
]

edges = [Edge("now", "diff", to_port="now"), Edge("last", "diff", to_port="before"),
         Edge("diff", "alert", from_port="changed"),
         Edge("diff", "quiet", from_port="same"),
         Edge("now", "save")]

bench = build("Watch for changes",
              "Check on a schedule, alert only when something moved.",
              stages, nodes, edges)
print("layers:", bench.layers())

problems: none
layers: [['now', 'last'], ['diff', 'save'], ['alert', 'quiet']]


In [3]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1100 308" width="1100" height="308" style="max-width:none" role="img"><defs><marker id="bg22958903-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0 · 2 parallel</text><g><rect x="60" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Read it now</text><text x="69" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="60" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">What we saw</text><text x="69" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="479.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1 · 2 parallel</text><g><rect x="386" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1" stroke-dasharray="6 3"/><text x="395" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Did it move?</text><text x="563" y="93.0" text-anchor="end" font-size="9" font-weight="700" fill="#2d6cb5">BRANCH</text><text x="395" y="108.0" font-size="9.5" fill="#68737f">1 candidate · one way out</text></g><g><rect x="386" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Remember it</text><text x="395" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="805.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2 · 2 parallel</text><g><rect x="712" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Raise an alert</text><text x="721" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="712" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Stay quiet</text><text x="721" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,100.0 C316.0,100.0 316.0,100.0 386,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg22958903-arrow)"/><text x="316.0" y="95.0" text-anchor="middle" font-size="9" fill="#68737f">now</text><path d="M246,182.0 C316.0,182.0 316.0,100.0 386,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg22958903-arrow)"/><text x="316.0" y="136.0" text-anchor="middle" font-size="9" fill="#68737f">before</text><path d="M572,100.0 C642.0,100.0 642.0,100.0 712,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg22958903-arrow)"/><text x="642.0" y="95.0" text-anchor="middle" font-size="9" fill="#68737f">changed</text><path d="M572,100.0 C642.0,100.0 642.0,182.0 712,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg22958903-arrow)"/><text x="642.0" y="136.0" text-anchor="middle" font-size="9" fill="#68737f">same</text><path d="M246,100.0 C316.0,100.0 316.0,182.0 386,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg22958903-arrow)"/></svg>', title='Watch for changes — shape', note='3 layers, widest 2. A dashed outline takes only one way out. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=308)

In [4]:
STATE = WORK / "last-seen.json"

WATCHED = {
    "run 1": {"version": "2.1.0", "status": "green"},
    "run 2": {"version": "2.1.0", "status": "green"},
    "run 3": {"version": "2.2.0", "status": "green"},
    "run 4": {"version": "2.2.0", "status": "red"},
}
current = {"value": None}

def read_now():
    return current["value"]

def read_last():
    return json.loads(STATE.read_text()) if STATE.exists() else {}

def diff_two(**kw):
    now, before = kw["now"], kw["before"]
    moved = {k: (before.get(k), v) for k, v in now.items() if before.get(k) != v}
    payload = {"changes": moved, "first_run": not before}
    return ("changed", payload) if moved else ("same", payload)

def alert_write(workspace, **kw):
    lines = [f"{k}: {old} -> {new}" for k, (old, new) in kw["in"]["changes"].items()]
    path = workspace / "alerts.log"
    with path.open("a") as handle:
        handle.write("\n".join(lines) + "\n")
    return {"alerted": lines}

def note_quiet(**kw):
    return {"alerted": []}

def save_state(**kw):
    STATE.write_text(json.dumps(kw["in"]))
    return {"saved": True}

runtime = execute.Runtime({
    "read.now": read_now, "read.last": read_last, "diff.two": diff_two,
    "alert.write": alert_write, "note.quiet": note_quiet, "save.state": save_state})

plan = compile_route(bench, {s.id: s.candidates[0] for s in bench.leaf_stages})

print(f"{'run':<8}{'reading':<32}{'path taken':<12}alerted")
for label, reading in WATCHED.items():
    current["value"] = reading
    got = execute.run(plan, runtime, workspace=WORK)
    taken = "alert" if not next(s for s in got.steps if s.stage == "alert").skipped else "quiet"
    alerted = got.values.get(("alert", "out"), {}).get("alerted", [])
    print(f"{label:<8}{str(reading):<32}{taken:<12}{alerted}")

run     reading                         path taken  alerted
run 1   {'version': '2.1.0', 'status': 'green'}alert       ['version: None -> 2.1.0', 'status: None -> green']
run 2   {'version': '2.1.0', 'status': 'green'}quiet       []
run 3   {'version': '2.2.0', 'status': 'green'}alert       ['version: 2.1.0 -> 2.2.0']
run 4   {'version': '2.2.0', 'status': 'red'}alert       ['status: green -> red']


Four runs, two alerts. Runs 2 and 4 read the same thing as the run before them
in one case and a real change in the other, and only the real changes made
noise.

The first run alerts because everything is new. That is a judgement call and it
is visible in `first_run`, so you can decide to suppress it rather than
discovering the behaviour later.

In [5]:
print((WORK / "alerts.log").read_text())

version: None -> 2.1.0
status: None -> green
version: 2.1.0 -> 2.2.0
status: green -> red

